# Day 9. Embedding

**오늘의 목표**
- Token ID를 그대로 쓰면 안 되는 이유를 설명할 수 있다
- `nn.Embedding`이 "표에서 행 하나 꺼내기(lookup)"라는 걸 안다
- Embedding이 사실 **one-hot @ Weight** = Bias 없는 Linear Layer라는 걸 확인한다
- `(B, T)` Token ID가 `(B, T, C)` 벡터로 바뀌는 과정을 shape로 따라간다
- 학습을 통해 Embedding에 **의미**가 생기는 걸 직접 본다

> 진행 방법: 위에서부터 셀을 하나씩 실행(`Shift + Enter`)하세요.
> `# TODO` 셀은 직접 채워보는 연습 문제입니다.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

plt.rcParams["font.family"] = "AppleGothic"
plt.rcParams["axes.unicode_minus"] = False
torch.manual_seed(0)

def r(t, d=3):
    # 텐서를 보기 좋게 반올림한 리스트로
    return [round(v, d) for v in t.tolist()]

## 1. 문제: Token ID는 그냥 "번호표"다

Day 8에서 Tokenizer가 문장을 숫자로 바꿨습니다.

```text
"나는 오늘 학교에 간다"  →  [0, 1, 2, 3]
```

이 숫자를 그대로 모델에 넣으면 안 될까요?

In [ ]:
vocab = ["나는", "오늘", "학교에", "간다", "고양이", "강아지"]
stoi = {w: i for i, w in enumerate(vocab)}
print(stoi)

**Token ID는 순서대로 붙인 번호표일 뿐**입니다.

- `고양이 = 4`, `강아지 = 5` → 번호가 가까워서 의미도 가까운 걸까요? **우연입니다.**
- `간다 = 3`, `고양이 = 4` → 번호는 1 차이지만 의미는 전혀 관계없어요.
- 실제 GPT-2에서 15342번과 15343번 Token은 서로 아무 상관이 없습니다.

숫자 하나로는 "고양이와 강아지는 비슷하고, 자동차와는 다르다" 같은 **여러 방향의 관계**를 담을 수 없어요.
그래서 Token마다 **벡터(숫자 여러 개)** 를 하나씩 줍니다. 이게 **Embedding**입니다.

```text
15342  →  Embedding  →  [0.13, -0.72, 0.44, ...]
```

## 2. 첫 시도: One-hot 벡터

가장 단순한 방법은 "자기 번호 자리만 1, 나머지는 0"인 벡터입니다.

In [ ]:
ids = torch.tensor([stoi[w] for w in vocab])
one_hot = F.one_hot(ids, num_classes=len(vocab)).float()

for w, row in zip(vocab, one_hot):
    print(f"{w:4s} → {row.int().tolist()}")

One-hot의 문제 두 가지:

**① 너무 길다**: 벡터 길이 = vocab_size. GPT-2는 50,257이라서 Token 하나에 숫자 5만 개, 그중 49,256개가 0이에요.

**② 모든 단어가 똑같이 남남이다**: Day 2에서 Dot Product로 "얼마나 비슷한지" 쟀죠? One-hot끼리는 **항상 0**입니다.

In [ ]:
cat, dog, go = one_hot[stoi["고양이"]], one_hot[stoi["강아지"]], one_hot[stoi["간다"]]
print("고양이 · 강아지 =", (cat @ dog).item())
print("고양이 · 간다   =", (cat @ go).item())

고양이-강아지나 고양이-간다나 똑같이 0입니다. **비슷함을 표현할 방법이 없어요.**

**원하는 것**: 짧고(예: 길이 8), 비슷한 단어끼리는 비슷한 방향을 가리키는 벡터

## 3. `nn.Embedding`: 거대한 표에서 행 하나 꺼내기

`nn.Embedding(vocab_size, C)`는 **`(vocab_size, C)` 크기의 표**를 하나 만듭니다.
Token ID가 들어오면 **그 번호의 행**을 꺼내줍니다.

```text
weight (vocab_size × C)
  0번 행: [ 0.12, -0.53, ...]   ← "나는"
  1번 행: [-0.81,  0.27, ...]   ← "오늘"
  2번 행: [ 0.44,  0.09, ...]   ← "학교에"
  ...
```

In [ ]:
vocab_size, C = len(vocab), 4
embedding = nn.Embedding(vocab_size, C)

print("weight shape:", tuple(embedding.weight.shape), " = (vocab_size, C)")
print(embedding.weight.data.round(decimals=2))

In [ ]:
token_id = torch.tensor(stoi["학교에"])
vec = embedding(token_id)

print("학교에 ID       :", token_id.item())
print("embedding(ID)   :", r(vec, 3))
print("weight[ID] 행   :", r(embedding.weight[token_id], 3))
print("같은가?", torch.equal(vec, embedding.weight[token_id]))

**정말 그냥 표에서 행을 꺼내는 것**뿐입니다. 계산이 거의 없어서 아주 빨라요.

### 3-1. 사실은 one-hot @ Weight 였다

Day 2, 3을 떠올려보세요. One-hot 벡터에 행렬을 곱하면?

```text
[0, 0, 1, 0, 0, 0]  @  weight (6 × 4)
     ↑ 2번 자리만 1
= weight의 2번 행
```

1이 있는 자리의 행만 살아남고 나머지는 0이 곱해져서 사라집니다.

In [ ]:
oh = F.one_hot(token_id, num_classes=vocab_size).float()   # (6,)
via_matmul = oh @ embedding.weight                          # (6,) @ (6, 4) → (4,)

print("one_hot @ weight :", r(via_matmul, 3))
print("embedding(ID)    :", r(embedding(token_id), 3))
print("같은가?", torch.allclose(via_matmul, embedding(token_id)))

> 💡 **Embedding = one-hot 입력에 Bias 없는 Linear Layer를 적용한 것**과 수학적으로 같습니다.
> 하지만 실제로 5만 칸짜리 one-hot을 만들고 곱하면 낭비라서, 바로 "행 꺼내기"로 계산하는 게 `nn.Embedding`이에요.

## 4. Shape: `(B, T)` → `(B, T, C)`

문장을 여러 개 한 번에 넣으면 Token ID는 `(B, T)` 모양입니다.
Embedding은 **각 ID를 길이 C 벡터로 바꾸니까** 마지막에 축이 하나 붙어요.

In [ ]:
sentences = [
    ["나는", "오늘", "학교에", "간다"],
    ["고양이", "오늘", "학교에", "간다"],
]
idx = torch.tensor([[stoi[w] for w in s] for s in sentences])

x = embedding(idx)
print("Token ID idx :", tuple(idx.shape), "  = (B, T)")
print(idx)
print()
print("embedding(idx):", tuple(x.shape), " = (B, T, C)")

In [ ]:
# 두 문장에서 "오늘"(같은 ID)은 똑같은 벡터가 됩니다
print("문장 0의 '오늘':", r(x[0, 1], 3))
print("문장 1의 '오늘':", r(x[1, 1], 3))
print("같은가?", torch.equal(x[0, 1], x[1, 1]))

> 💡 같은 Token은 **문장 어디에 있든 항상 같은 벡터**가 됩니다.
> "오늘"이 첫 번째에 있는지 다섯 번째에 있는지는 Embedding만으로는 알 수 없어요. → 내일 Day 10 **Positional Encoding**이 이 문제를 해결합니다.

## 5. Embedding 값은 누가 정하나? → 학습!

Day 3의 Weight와 똑같습니다. 처음엔 **랜덤**이고, 학습하면서 바뀝니다.

In [ ]:
print("requires_grad:", embedding.weight.requires_grad)
print("파라미터 개수:", embedding.weight.numel(), "=", vocab_size, "×", C)

### 5-1. Cosine Similarity: 방향이 얼마나 비슷한가

Day 2의 Dot Product는 벡터 **길이**에도 영향을 받습니다. 방향만 비교하려면 길이로 나눠줍니다.

```text
cos_sim(a, b) = (a · b) / (|a| × |b|)
```

- `1` : 완전히 같은 방향
- `0` : 직각 (관계 없음)
- `-1`: 정반대 방향

In [ ]:
a = torch.tensor([1.0, 2.0, 3.0])
print("a vs a      :", round(F.cosine_similarity(a, a, dim=0).item(), 3))
print("a vs 2a     :", round(F.cosine_similarity(a, 2 * a, dim=0).item(), 3), " ← 길이만 다르고 방향은 같음")
print("a vs -a     :", round(F.cosine_similarity(a, -a, dim=0).item(), 3))
print("a vs [3,0,-1]:", round(F.cosine_similarity(a, torch.tensor([3.0, 0.0, -1.0]), dim=0).item(), 3))

### 5-2. 실험: 학습하면 의미가 생길까?

단어 9개를 3그룹으로 나눕니다. 모델은 이 그룹을 **모릅니다.**

| 동물 | 과일 | 탈것 |
|---|---|---|
| 고양이, 강아지, 토끼 | 사과, 바나나, 포도 | 자동차, 버스, 기차 |

과제: 단어 하나를 보고 **"같은 문장에 같이 나올 법한 단어"** 를 맞히기.
(학습 데이터에서 같은 그룹 단어끼리만 같이 등장하도록 만들어요.)

모델 구조: `Embedding(9, 8)` → `Linear(8, 9)` → Cross Entropy (Day 6!)

In [ ]:
words = ["고양이", "강아지", "토끼", "사과", "바나나", "포도", "자동차", "버스", "기차"]
group = torch.tensor([0, 0, 0, 1, 1, 1, 2, 2, 2])
V, C = len(words), 8

def make_batch(n=256):
    # 단어 하나와, 같은 그룹의 '같이 등장한 단어' 하나를 랜덤으로 뽑기
    center = torch.randint(0, V, (n,))
    same_group = [torch.nonzero(group == group[c]).flatten() for c in center]
    context = torch.stack([g[torch.randint(0, len(g), (1,))].squeeze() for g in same_group])
    return center, context

torch.manual_seed(42)
emb = nn.Embedding(V, C)
head = nn.Linear(C, V)
before = emb.weight.detach().clone()          # 학습 전 Embedding 저장

opt = torch.optim.AdamW(list(emb.parameters()) + list(head.parameters()), lr=0.05)
for step in range(300):
    center, context = make_batch()
    logits = head(emb(center))                # (n, C) → (n, V)
    loss = F.cross_entropy(logits, context)
    opt.zero_grad()
    loss.backward()
    opt.step()

after = emb.weight.detach().clone()
print(f"최종 loss: {loss.item():.3f}   (그룹 안 3단어 중 랜덤이라 이론상 최소 log(3) = {torch.log(torch.tensor(3.0)).item():.3f})")

In [ ]:
def cos_matrix(W):
    Wn = F.normalize(W, dim=-1)       # 각 행의 길이를 1로
    return Wn @ Wn.T                  # 모든 쌍의 cosine similarity

fig, axes = plt.subplots(1, 2, figsize=(13, 5.2))
for ax, W, title in [(axes[0], before, "학습 전 (랜덤)"), (axes[1], after, "학습 후")]:
    S = cos_matrix(W)
    im = ax.imshow(S, cmap="RdBu_r", vmin=-1, vmax=1)
    ax.set_xticks(range(V), words, rotation=45)
    ax.set_yticks(range(V), words)
    ax.set_title(f"Cosine Similarity: {title}", fontsize=13)
    for i in range(V):
        for j in range(V):
            ax.text(j, i, f"{round(S[i, j].item(), 1) + 0.0:.1f}", ha="center", va="center", fontsize=8,
                    color="white" if abs(S[i, j]) > 0.6 else "black")
fig.colorbar(im, ax=axes, shrink=0.8)
plt.show()

- **학습 전**: 색이 뒤죽박죽. 고양이-강아지가 고양이-버스보다 가깝다는 근거가 없어요.
- **학습 후**: 대각선을 따라 **3×3 빨간 블록 3개**가 생겼습니다. 같은 그룹끼리는 비슷한 방향, 다른 그룹과는 먼 방향이 됐어요.

**아무도 "고양이와 강아지는 비슷하다"고 알려주지 않았습니다.**
"비슷한 맥락에서 등장한다"는 데이터만으로 Embedding이 스스로 의미를 배웠어요. 실제 LLM도 똑같은 원리입니다.

## 6. Embedding Dimension(C)은 얼마나 크게?

| C | 장점 | 단점 |
|---|---|---|
| 작게 (예: 32) | 파라미터 적음, 빠름 | 담을 수 있는 의미의 "방향"이 적음 |
| 크게 (예: 768) | 미묘한 의미 차이까지 표현 | 파라미터 많음, 학습 데이터도 많이 필요 |

실제 모델 크기:

In [ ]:
for name, vocab_n, c in [
    ("4주차 mini GPT (글자 단위)", 65, 32),
    ("GPT-2 small", 50257, 768),
    ("GPT-2 XL", 50257, 1600),
]:
    params = vocab_n * c
    print(f"{name:26s} vocab={vocab_n:6d}, C={c:5d} → Embedding 파라미터 {params:>12,d}개")

GPT-2 small 전체 파라미터가 약 1.24억 개인데, Token Embedding 하나가 **약 3,860만 개 (약 31%)** 를 차지합니다.

## 7. LLM과의 연결

Day 8 Tokenizer부터 오늘 Embedding까지 이어 붙이면:

```text
"나는 오늘 학교에 간다"
   ↓ Tokenizer (Day 8)
[0, 1, 2, 3]                      (T,)  → batch로 묶으면 (B, T)
   ↓ token_embedding (오늘)
(B, T, C) 벡터                     ← Transformer가 받는 입력
   ↓ + Position 정보 (Day 10)
   ↓ Transformer Block × N
   ↓ lm_head: Linear(C, vocab_size) (Day 3)
(B, T, vocab_size) 점수            → Softmax (Day 5) → 다음 Token
```

커리큘럼 6-3에서 구현할 코드가 바로 이것입니다.

In [ ]:
B, T, n_embd, vocab_size = 2, 4, 32, 65

token_embedding = nn.Embedding(vocab_size, n_embd)
lm_head = nn.Linear(n_embd, vocab_size, bias=False)

idx = torch.randint(0, vocab_size, (B, T))
x = token_embedding(idx)
logits = lm_head(x)                       # (중간의 Transformer Block은 3주차에)

print("idx    :", tuple(idx.shape))
print("x      :", tuple(x.shape))
print("logits :", tuple(logits.shape))

### 7-1. 미리보기: Weight Tying

`token_embedding.weight`는 `(vocab_size, C)`, `lm_head.weight`도 `(vocab_size, C)`입니다. (Day 3: PyTorch는 `(out, in)`으로 저장)
**shape가 똑같죠?**

- Embedding: Token ID → 벡터 ("이 단어는 이런 의미야")
- lm_head: 벡터 → 각 Token 점수 ("이 의미면 어떤 단어일까")

서로 반대 방향의 작업이라, GPT-2는 **두 Weight를 하나로 공유**합니다. 파라미터가 3,860만 개 줄어들어요.

In [ ]:
print("token_embedding.weight:", tuple(token_embedding.weight.shape))
print("lm_head.weight        :", tuple(lm_head.weight.shape))

lm_head.weight = token_embedding.weight      # 같은 텐서를 가리키게
print("공유 후 같은 메모리인가?", lm_head.weight.data_ptr() == token_embedding.weight.data_ptr())

## 8. 연습 문제

`assert`가 에러 없이 통과하면 정답입니다.

**Q1.** 실행하기 **전에** 결과 shape를 예측해서 튜플로 적으세요.

In [ ]:
emb = nn.Embedding(100, 16)
idx = torch.randint(0, 100, (3, 10))

weight_shape = "?"   # TODO: emb.weight의 shape
out_shape    = "?"   # TODO: emb(idx)의 shape

assert weight_shape == tuple(emb.weight.shape), "weight shape를 다시 생각해보세요"
assert out_shape == tuple(emb(idx).shape), "출력 shape를 다시 생각해보세요"
print("Q1 정답!")

**Q2.** `emb(...)`를 호출하지 않고, `emb.weight`에서 **직접 행을 꺼내서** 같은 결과를 만드세요.
힌트: 텐서 인덱싱에 텐서를 넣을 수 있어요. (`W[ids]`)

In [ ]:
emb = nn.Embedding(10, 4)
ids = torch.tensor([[3, 7, 7], [0, 1, 9]])

manual = None  # TODO

assert manual is not None and manual.shape == emb(ids).shape, "shape가 달라요"
assert torch.equal(manual, emb(ids)), "값이 달라요"
print("Q2 정답!", tuple(manual.shape))

**Q3.** `F.cosine_similarity`를 쓰지 않고 Cosine Similarity를 직접 구현하세요.
힌트: `a @ b`, `a.norm()` (벡터 길이)

In [ ]:
def my_cos_sim(a, b):
    return None  # TODO

assert my_cos_sim(torch.randn(8), torch.randn(8)) is not None, "함수를 완성해주세요"
for _ in range(5):
    a, b = torch.randn(8), torch.randn(8)
    assert torch.allclose(torch.as_tensor(my_cos_sim(a, b)), F.cosine_similarity(a, b, dim=0), atol=1e-6), "공식을 다시 확인해보세요"
print("Q3 정답!")

**Q4.** 한국어 모델을 만든다고 합시다. vocab_size = 32,000, C = 512입니다.
- Token Embedding 파라미터 개수는?
- 7-1처럼 lm_head와 Weight를 공유하지 **않으면**, Embedding + lm_head(bias 없음) 파라미터 합은?

In [ ]:
vocab_size, C = 32000, 512

emb_params      = "?"   # TODO: 숫자
untied_total    = "?"   # TODO: 숫자

emb = nn.Embedding(vocab_size, C)
head = nn.Linear(C, vocab_size, bias=False)
assert emb_params == emb.weight.numel(), "Embedding 파라미터 개수를 다시 계산해보세요"
assert untied_total == emb.weight.numel() + head.weight.numel(), "합계를 다시 계산해보세요"
print(f"Q4 정답! 공유하면 {head.weight.numel():,}개를 아낄 수 있어요")

## 9. 체크 질문 (코드 없이 말로 답해보기)

답을 아래 셀에 적어두거나 채팅으로 알려주세요.

1. Token ID를 그대로 모델에 넣으면 왜 안 되는가?
2. One-hot 벡터의 문제 두 가지는?
3. `nn.Embedding`은 내부에서 무슨 일을 하는가? Linear Layer와 어떤 관계인가?
4. Token ID와 Embedding의 차이는 무엇인가? (커리큘럼 최종 질문)
5. Embedding 값은 처음에 어떻게 정해지고, 어떻게 "의미"를 갖게 되는가?
6. 같은 Token이 문장의 다른 위치에 있으면 Embedding은 어떻게 되는가? 이게 왜 문제인가?

**내 답:**

1.
2.
3.
4.
5.
6.